# Evolution Plots

In [1]:
import diffusers
from transformers import CLIPVisionModelWithProjection, CLIPImageProcessor
import torch
import cv2
import numpy as np
import os
import random
from tqdm import tqdm
from PIL import Image 

/home/sonia/miniconda3/envs/svd/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# -------- helper to draw black outline & hide ticks --------
def outline_axes(ax, color='k', lw=1.0):
    # ensure frame visible
    ax.set_frame_on(True)
    # black spines
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor(color)
        spine.set_linewidth(lw)
    # no ticks / labels
    ax.tick_params(
        axis='both',
        which='both',
        bottom=False, top=False, left=False, right=False,
        labelbottom=False, labelleft=False
    )

In [3]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import numpy as np

expname = 'sanity'
os.makedirs(expname, exist_ok=True)

# sid = '19610215000445016000'
realsetpath = '/home/cyclone/train/multivar/0.25/date/natlantic/val/'
synthsetpath = '/mnt/data/sonia/geodes-samples/date/train3d.1e-7.img_multivar_date_2d_1e-7_5737618_20_5741209/samples'
sids = os.listdir(synthsetpath)
for sid in sids:
    fig = plt.figure(figsize=(7, 8), constrained_layout=True)
    # fig.suptitle(sid)
    fig.set_constrained_layout_pads(w_pad=0, h_pad=0.1, hspace=0, wspace=0)
    subfigs = fig.subfigures(2, 1, hspace=0.03)

    realpath = os.path.join(realsetpath, sid)
    synthpath = os.path.join(synthsetpath, sid)

    # Load real storm data
    real = []
    for i in range(8):
        img = np.load(f'{realpath}/{i}.npy')  
        img = np.sqrt(img[:,:,1]**2 + img[:,:,2]**2)
        real.append(img)
        
    # Load synth storm data
    synth = []
    for i in range(8):
        img = np.load(f'{synthpath}/{i}.npy').squeeze()
        # img = img.astype(np.float32)
        # img = np.expm1(img)
        img = np.sqrt(img[:,:,1]**2 + img[:,:,2]**2)
        synth.append(img)
    maxp = max([np.max(p) for p in synth])
    minp = min([np.min(p) for p in synth])
        
    # 1. Define discrete levels and colormap
    levels = list(range(0,100, 10))
    # Start from a discrete viridis
    base = plt.cm.get_cmap('viridis', len(levels) - 1)
    colors = base(np.arange(base.N))

    # Make the first bin (0–10) white
    colors[0] = (1.0, 1.0, 1.0, 1.0)  # RGBA white
    colors[1] = (0.73, 0.87, 1.00, 1.0) 
    colors[2] = (0.50, 0.75, 1.00, 1.0) 
    colors[3] = (0.31, 0.63, 0.96, 1.0) 
    colors[4] = (0.12, 0.61, 0.80, 1.0) 
    colors[5] = (0.00, 0.68, 0.54, 1.0) 
    colors[6] = (0.42, 0.75, 0.29, 1.0) 
    colors[7] = (0.78, 0.84, 0.17, 1.0) 
    colors[8] = (1.00, 0.85, 0.12, 1.0) 

    # Build a ListedColormap with modified colors
    cmap = mcolors.ListedColormap(colors)
    norm = mcolors.BoundaryNorm(boundaries=levels, ncolors=cmap.N)

    # real subplot
    subfigs[0].suptitle('Real Cyclone-Centered Wind Magnitude', fontsize=16, fontweight='bold')
    # Create 2x4 grid inside the top subfigure
    axes = subfigs[0].subplots(2, 4, sharex=True, sharey=True, 
                            gridspec_kw={'wspace': 0, 'hspace': 0})
    axes = axes.flatten()  # Flatten to simplify indexing

    # plot real data
    for i in range(8):
        img = real[i]
        img = img - minp 
        img = img / (maxp - minp)  # Normalize the image data
        img = 100*img
        img = img.astype(np.uint8)
        axes[i].imshow(img, cmap=cmap, norm=norm)
        axes[i].set_title(f'{i*6} Hours')
        outline_axes(axes[i])
        

    # synth subplot
    subfigs[1].suptitle('Synthetic Cyclone-Centered Wind Magnitude', fontsize=16, fontweight='bold')
    # Create 2x4 grid inside the top subfigure
    axes = subfigs[1].subplots(2, 4, sharex=True, sharey=True, 
                            gridspec_kw={'wspace': 0, 'hspace': 0})
    axes = axes.flatten()  # Flatten to simplify indexing

    # plot synth data
    for i in range(8):
        img = synth[i]
        img = img - minp 
        img = img / (maxp - minp)  # Normalize the image data
        img = 100*img
        img = img.astype(np.uint8)
        axes[i].imshow(img, cmap=cmap, norm=norm)
        axes[i].set_title(f'{i*6} Hours')
        outline_axes(axes[i])


    real_levels = np.linspace(0, maxp, len(levels))
    norm_unnormalized = mcolors.BoundaryNorm(boundaries=real_levels, ncolors=cmap.N)
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm_unnormalized)
    sm.set_array([])  # Dummy array is required for the ScalarMappable to initialize
    # add_axes defines [left, bottom, width, height] in figure coordinates (0 to 1):
    cbar_ax = fig.add_axes([0.03, -0.05, 0.94, 0.02]) 
    cbar = fig.colorbar(sm, cax=cbar_ax, orientation='horizontal')
    cbar.ax.xaxis.set_major_formatter(plt.FormatStrFormatter('%.1f'))
    cbar.set_label('Wind Magnitude (m/s)', fontsize=12)

    plt.savefig(f'{expname}/fig_{sid}.png', bbox_inches='tight', pad_inches=0)
    plt.close()

/tmp/ipykernel_315920/3286700421.py:42: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  base = plt.cm.get_cmap('viridis', len(levels) - 1)
